<h1 style="text-align: center;">
    <div style="color: #808080; font-size: 60%">"Who Gets the H1B? Analyzing 2025 Sponsor, Salary, and Job Title Trends"</div>
    <div style="color:rgb(238, 167, 26)"> H1B 2025 Data Analysis and SQL Exploration</div>
    <!-- <div style="color:rgb(163, 172, 107);font-size: 40%"> color trying1</div>
    <div style="color:rgb(128, 205, 180);font-size: 40%"> color trying1</div>
    <div style="color:rgb(51, 36, 185);font-size: 40%"> color trying1</div> -->
    <div style="font-size: 60%;"><a href="https://www.dol.gov/agencies/eta/foreign-labor/performance">
    Download the U.S. Department of Labor H1B Disclosure Data</a></div>

<hr style="color:rgb(204, 125, 105);">

# The introduction of this analysis

Education equips people with skills to contribute to society — and for many students, the next step after graduation is employment. However, for international students in the U.S., securing a job is not just about ability — it also requires legal work authorization after the F-1 visa and OPT period expire. The H-1B visa allows U.S. companies to hire foreign workers in specialty occupations. 

This project aims to help international students better understand the H-1B landscape :
- Which positions are most commonly sponsored? 
- Which employers submit the most applications?
- How salaries and approval rates vary?
...

so we can make more informed career decisions.

This notebook uses SQL for querying and Python (pandas, matplotlib/plotly) for visualizing and interpreting the results.


# Section 1: Data Clean
We should always clean and prepare the dataset. This includes handle missing and irrelevant values, transforming the data into a structure better for analysis.

In [53]:
import pandas as pd

file_path = "/Users/zhoujingya/Documents/school project/UIUC🌽/projects/H1BView/data/LCA_Disclosure_Data_FY2025_Q2.xlsx"

df = pd.read_excel(file_path)

df.head()


,CASE_NUMBER,CASE_STATUS,RECEIVED_DATE,DECISION_DATE,ORIGINAL_CERT_DATE,VISA_CLASS,JOB_TITLE,SOC_CODE,SOC_TITLE,FULL_TIME_POSITION,...,WILLFUL_VIOLATOR,SUPPORT_H1B,STATUTORY_BASIS,APPENDIX_A_ATTACHED,PUBLIC_DISCLOSURE,PREPARER_LAST_NAME,PREPARER_FIRST_NAME,PREPARER_MIDDLE_INITIAL,PREPARER_BUSINESS_NAME,PREPARER_EMAIL
0,I-200-25090-814321,Withdrawn,2025-03-31,2025-03-31,NaT,H-1B,Director - Software Development,15-1252.00,Software Developers,N,...,No,NaN,NaN,NaN,Disclose Business,Esily,Hady,NaN,Ray Law International P.C.,raylawcases@raylawinternational.com
1,I-200-25090-815325,Withdrawn,2025-03-31,2025-03-31,NaT,H-1B,IT Consultant(Peoplesoft Developer),15-1252.00,Software Developers,Y,...,No,Yes,"$60,000 or higher annual wage",NaN,Disclose Business,NaN,NaN,NaN,NaN,NaN
2,I-200-25090-814481,Withdrawn,2025-03-31,2025-03-31,NaT,H-1B,Software Developer,15-1252.00,Software Developers,Y,...,No,NaN,NaN,NaN,Disclose Business,NaN,NaN,NaN,NaN,NaN
3,I-200-25090-813856,Withdrawn,2025-03-31,2025-03-31,NaT,H-1B,Assistant Professor,25-1071.00,"Health Specialties Teachers, Postsecondary",Y,...,No,NaN,NaN,NaN,Disclose Business,NaN,NaN,NaN,NaN,NaN
4,I-200-25090-813621,Withdrawn,2025-03-31,2025-03-31,NaT,H-1B,Enterprise Resource Planning Advisor,15-1253.00,Software Quality Assurance Analysts and Testers,Y,...,No,NaN,NaN,NaN,Disclose Business,Wills,Stacy,NaN,Berry Appleman and Leiden LLP,swills@bal.com


I reorganized the data, selected the relevant columns, and renamed them into a more readable form.

In [54]:
df = df[[
    "EMPLOYER_NAME",
    "JOB_TITLE",
    "WORKSITE_STATE",
    "WORKSITE_CITY",
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_RATE_OF_PAY_TO",
    "WAGE_UNIT_OF_PAY",
    "CASE_STATUS",
    "FULL_TIME_POSITION",
    "SOC_TITLE"
]]

df = df.rename(columns={
    "EMPLOYER_NAME": "employer",
    "JOB_TITLE": "job_title",
    "WORKSITE_STATE": "state",
    "WORKSITE_CITY": "city",
    "WAGE_RATE_OF_PAY_FROM": "salary_from",
    "WAGE_RATE_OF_PAY_TO": "salary_to",
    "WAGE_UNIT_OF_PAY": "salary_unit",
    "CASE_STATUS": "case_status",
    "FULL_TIME_POSITION": "full_time",
    "SOC_TITLE": "soc_title"
})


df



,employer,job_title,state,city,salary_from,salary_to,salary_unit,case_status,full_time,soc_title
0,BioData Solutions,Director - Software Development,PA,Malvern,60.41,NaN,Hour,Withdrawn,N,Software Developers
1,"People Tech Group, Inc",IT Consultant(Peoplesoft Developer),IL,Lake Villa,127754.00,127755.0,Year,Withdrawn,Y,Software Developers
2,FutureTech Consultants LLC,Software Developer,GA,Peachtree Corners,105227.00,NaN,Year,Withdrawn,Y,Software Developers
3,Texas A&M University,Assistant Professor,TX,College Station,125000.00,NaN,Year,Withdrawn,Y,"Health Specialties Teachers, Postsecondary"
4,"NTT DATA Americas, Inc.",Enterprise Resource Planning Advisor,NJ,Morris Plains,138176.00,168176.0,Year,Withdrawn,Y,Software Quality Assurance Analysts and Testers
...,...,...,...,...,...,...,...,...,...,...
240765,Snowflake Inc.,SENIOR SALESFORCE DEVELOPER,CA,DUBLIN,180000.00,NaN,Year,Certified - Withdrawn,Y,Computer Systems Analysts
240766,UNIVERSITY OF DELAWARE,Post Doctoral Researcher (Behavioral Health & ...,DE,Newark,54540.00,NaN,Year,Certified - Withdrawn,Y,Food Scientists and Technologists
240767,Georgia State University,Postdoctoral Research Associate,GA,Atlanta,48000.00,NaN,Year,Certified - Withdrawn,Y,Biochemists and Biophysicists
240768,UNIVERSITY OF DELAWARE,Post Doctoral Researcher (Behavioral Health & ...,DE,Newark,53076.00,NaN,Year,Certified - Withdrawn,Y,Food Scientists and Technologists


In [56]:
df["employer"] = df["employer"].str.strip().str.lower()
df

,employer,job_title,state,city,salary_from,salary_to,salary_unit,case_status,full_time,soc_title
0,biodata solutions,Director - Software Development,PA,Malvern,60.41,NaN,Hour,Withdrawn,N,Software Developers
1,"people tech group, inc",IT Consultant(Peoplesoft Developer),IL,Lake Villa,127754.00,127755.0,Year,Withdrawn,Y,Software Developers
2,futuretech consultants llc,Software Developer,GA,Peachtree Corners,105227.00,NaN,Year,Withdrawn,Y,Software Developers
3,texas a&m university,Assistant Professor,TX,College Station,125000.00,NaN,Year,Withdrawn,Y,"Health Specialties Teachers, Postsecondary"
4,"ntt data americas, inc.",Enterprise Resource Planning Advisor,NJ,Morris Plains,138176.00,168176.0,Year,Withdrawn,Y,Software Quality Assurance Analysts and Testers
...,...,...,...,...,...,...,...,...,...,...
240765,snowflake inc.,SENIOR SALESFORCE DEVELOPER,CA,DUBLIN,180000.00,NaN,Year,Certified - Withdrawn,Y,Computer Systems Analysts
240766,university of delaware,Post Doctoral Researcher (Behavioral Health & ...,DE,Newark,54540.00,NaN,Year,Certified - Withdrawn,Y,Food Scientists and Technologists
240767,georgia state university,Postdoctoral Research Associate,GA,Atlanta,48000.00,NaN,Year,Certified - Withdrawn,Y,Biochemists and Biophysicists
240768,university of delaware,Post Doctoral Researcher (Behavioral Health & ...,DE,Newark,53076.00,NaN,Year,Certified - Withdrawn,Y,Food Scientists and Technologists


Then, I standardized the columns relate to `salary` to ensure all the values have the consistent units.

- Converted hourly wages to annual salary

- Renamed and calculated a new estimated_yearly_salary column

In [57]:
df = df.dropna(subset=["employer", "job_title", "state", "city", "salary_from", "salary_unit", "case_status"])
df["salary_to"] = df["salary_to"].fillna(df["salary_from"])
df["salary_unit"] = df["salary_unit"].str.upper()
df = df[df["salary_unit"] == "YEAR"]

In [58]:
# convert all salary unit to year
def convert_to_yearly(row):
    if row["salary_unit"] == "YEAR":
        return (row["salary_from"] + row["salary_to"]) / 2
    elif row["salary_unit"] == "HOUR":
        return ((row["salary_from"] + row["salary_to"]) / 2) * 2080
    elif row["salary_unit"] == "WEEK":
        return ((row["salary_from"] + row["salary_to"]) / 2) * 52
    elif row["salary_unit"] == "MONTH":
        return ((row["salary_from"] + row["salary_to"]) / 2) * 12
    else:
        return None
    
    
df["estimated_yearly_salary"] = df.apply(convert_to_yearly, axis=1)
df = df.drop_duplicates()
df

,employer,job_title,state,city,salary_from,salary_to,salary_unit,case_status,full_time,soc_title,estimated_yearly_salary
1,"people tech group, inc",IT Consultant(Peoplesoft Developer),IL,Lake Villa,127754.0,127755.0,YEAR,Withdrawn,Y,Software Developers,127754.5
2,futuretech consultants llc,Software Developer,GA,Peachtree Corners,105227.0,105227.0,YEAR,Withdrawn,Y,Software Developers,105227.0
3,texas a&m university,Assistant Professor,TX,College Station,125000.0,125000.0,YEAR,Withdrawn,Y,"Health Specialties Teachers, Postsecondary",125000.0
4,"ntt data americas, inc.",Enterprise Resource Planning Advisor,NJ,Morris Plains,138176.0,168176.0,YEAR,Withdrawn,Y,Software Quality Assurance Analysts and Testers,153176.0
5,arizona state university,Assistant Professor,AZ,Tempe,79760.0,79760.0,YEAR,Withdrawn,Y,"Art, Drama, and Music Teachers, Postsecondary",79760.0
...,...,...,...,...,...,...,...,...,...,...,...
240765,snowflake inc.,SENIOR SALESFORCE DEVELOPER,CA,DUBLIN,180000.0,180000.0,YEAR,Certified - Withdrawn,Y,Computer Systems Analysts,180000.0
240766,university of delaware,Post Doctoral Researcher (Behavioral Health & ...,DE,Newark,54540.0,54540.0,YEAR,Certified - Withdrawn,Y,Food Scientists and Technologists,54540.0
240767,georgia state university,Postdoctoral Research Associate,GA,Atlanta,48000.0,48000.0,YEAR,Certified - Withdrawn,Y,Biochemists and Biophysicists,48000.0
240768,university of delaware,Post Doctoral Researcher (Behavioral Health & ...,DE,Newark,53076.0,53076.0,YEAR,Certified - Withdrawn,Y,Food Scientists and Technologists,53076.0


At last, I save the cleaned data into a new csv

In [59]:
df.to_csv("h1b_clean_2025.csv", index=False)

# Section 2: Analysis

After cleaning the dataset, I loaded it into a local SQLite database to enable efficient SQL querying and relational-style analysis.

In [60]:
import sqlite3

df = pd.read_csv("h1b_clean_2025.csv")
# construct the SQLite data lib
conn = sqlite3.connect("h1b_data.db")
df.to_sql("h1b_2025", conn, if_exists="replace", index=False)

191669

## 1. Explore the top 10 companies sponsoring the most h1b applications

In [61]:
query_sponsor = """
SELECT employer,  
COUNT(*) AS total_cases,
salary_from,
salary_to
FROM h1b_2025
GROUP BY employer
ORDER BY total_cases DESC
LIMIT 10;
"""

top_sponsor = pd.read_sql(query_sponsor, conn)
top_sponsor

,employer,total_cases,salary_from,salary_to
0,ernst & young u.s. llp,5621,115000.08,115000.08
1,amazon.com services llc,5278,146307.00,146307.00
2,microsoft corporation,3234,159300.00,159300.00
3,google llc,2902,220182.00,220182.00
4,apple inc.,2471,156770.00,234700.00
5,cognizant technology solutions us corp,2279,126000.00,126000.00
6,"meta platforms, inc",2225,211571.00,211571.00
7,tata consultancy services limited,1686,62338.00,126700.00
8,"amazon web services, inc.",1612,180000.00,180000.00
9,deloitte consulting llp,1444,155800.00,155800.00


In [74]:
import plotly.express as px
top_sponsor["salary_midpoint"] = (top_sponsor["salary_from"] + top_sponsor["salary_to"]) / 2
fig = px.bar(
    top_sponsor.sort_values("total_cases", ascending=True),  # horizontal
    x="total_cases",
    y="employer",
    color="salary_midpoint",
    orientation="h",
    color_continuous_scale="Plasma",  
    labels={"total_cases": "H1B Sponsorship Count", "employer": "Employer", "salary_midpoint": "Avg Salary"},
    title="Top H1B Sponsors in 2025: Volume vs. Salary"
)

fig.show()

From the result, we observe that large companies are more likely to sponsor employees. These employers generally fall into two categories:

- Tech giants: Amazon, Microsoft, Google, Apple, Meta, AWS
- Consulting firms: Ernst & Young, Cognizant, Tata, Deloitte

Without deeper role-level analysis, we can broadly catch:
- Meta and google offer the highest salary
- apple shows a relatively wide range of salary, possibly due to the various positions or levels
- On average, tech companies tend to offer higher salaries than consulting companies.

## 2. The most popular sponsored job title in each state

In [86]:
hot = """ 
SELECT * 
FROM (
SELECT state, job_title, total,
    RANK() OVER (PARTITION BY state ORDER BY total DESC) AS rnk
FROM (
    SELECT state, job_title, COUNT(*) AS total
    FROM h1b_2025
    WHERE case_status = 'Certified' 
    GROUP BY state, job_title
    )
)
WHERE rnk <= 3
"""

h = pd.read_sql(hot, conn)
h

,state,job_title,total,rnk
0,AK,Elementary Teacher,8,1
1,AK,Certified Teacher,5,2
2,AK,Teacher,5,2
3,AL,Assistant Professor,33,1
4,AL,Postdoctoral Fellow,12,2
...,...,...,...,...
251,WY,Senior Test Engineer,1,3
252,WY,Software DevOps & Elastic Search Engineer,1,3
253,WY,Software Developer,1,3
254,WY,Technical Program Management,1,3


In [87]:
q = """ 
SELECT state, job_title, salary_to, salary_from, COUNT(job_title)
FROM h1b_2025
WHERE case_status = 'Certified' and job_title LIKE '%software%'
GROUP BY state
"""

hq = pd.read_sql(q, conn)
hq

,state,job_title,salary_to,salary_from,COUNT(job_title)
0,AL,Software QA Automation Engineer,74049.00,74049.00,50
1,AR,"Senior, Software Engineer",145000.00,145000.00,595
2,AZ,Principal Software Engineer,191280.00,151757.00,654
3,CA,Senior Embedded Software Engineer,165000.00,165000.00,9152
4,CO,Software Development Engineer,150000.00,150000.00,462
5,CT,Senior Software Engineer 1,199525.00,147475.00,264
6,DC,Manager JC50 - Software Developers,150500.00,150500.00,99
7,DE,"Vice President, Sr Manager of Software Enginee...",201400.00,201400.00,247
8,FL,"Associate, Software Engineer III",135000.00,135000.00,982
9,GA,"Principal Engineer, Software",178595.15,178595.15,1669


In [78]:
fig = px.treemap(
    h[h["rnk"] <= 3],
    path=["state", "job_title"],
    values="total",
    color="total",
    color_continuous_scale="Viridis",
    title="Top 3 Sponsored Job Titles per State"
)
fig.show()

In [146]:
fig = px.bar(
    h.sort_values("total", ascending=True),
    x="total",
    y="state",
    color="job_title",
    orientation="h",
    title="Most Sponsored Job Title per State (2025)",
    width=800,
    height=600
)
fig.show()

From the distribution of the job type among states, we can conclude that:
- The most frequently sponsored positions across states are overwhelmingly software-oriented, including Software Developer, Software Engineer, and Senior Software Engineer
- TX, CA, and WA have the most sponsored job opportunities, which are all tech hubs
- Besides engineers, positions like associate professors also appears in several states, suggesting the academic and research path for securing h1b

Titles like Senior Software Engineer, roles coming up with higher expectations, show up frequently. This suggests a tougher competition in landing h1b opportunities.

## 3. Compare the relative wages for poplar sponsored jobs among companies

In [111]:
# meadian wages

wage = """
SELECT employer AS sponsor,
job_title,
COUNT(*) AS number_of_employees,
AVG((salary_to + salary_from) * 1.0 / 2) AS avg_salary
FROM h1b_2025
WHERE LOWER(job_title) LIKE '%software%' AND case_status = 'Certified'
GROUP BY employer, job_title
HAVING number_of_employees > 20
ORDER BY AVG((salary_to + salary_from) * 1.0 / 2) DESC
LIMIT 20
"""

wages = pd.read_sql(wage, conn)
wages


,sponsor,job_title,number_of_employees,avg_salary
0,"fidelity technology group, llc d/b/a fidelity ...",Senior Software Engineer/Developer,23,431666.565217
1,apple inc.,Software Development Engineering,114,282496.049430
2,"salesforce, inc.",Software Engineering PMTS,42,277904.629762
3,"meta platforms, inc",Software Engineering Manager,62,270751.699355
4,google llc,Staff Software Engineer,25,266340.000000
5,adobe inc.,Software Development Engineer,236,246829.769195
6,"uber technologies, inc.",Staff Software Engineer,24,243233.333333
7,snowflake inc.,Senior Software Engineer,24,243135.000000
8,"salesforce, inc.",Software Engineering LMTS,110,230910.601909
9,bytedance inc.,Software Engineer,45,230458.316667


In [121]:
salary = px.bar(wages, x= wages.sponsor, y=wages.avg_salary, color='job_title', title='sponsored job salary comparison')
salary

The chart highlights companies offering the highest salaries for software-related roles. Since many of the job titles shown are mid-to-senior-level positions, the results reflect the upper bounds of compensation in the software engineering field. 

Particularly, one company stands out. Fidelity Investments, though a financial services company, maintains a strong tech presence and actively sponsors high-paying software positions, making it a promising yet less obvious option for traditional tech engineers.

Above all, those insights are more valuable for experienced professionals considering a job change. On the other hand, for entry-level job seekers, due to the lack of standardized job level tags in the dataset, this analysis cannot accurately separate entry-level from senior-level positions. 

# Summarization

In this project, I explored and analyzed the the second quarter of 2025 H-1B dataset with a focus on job title, employer sponsorship, and estimated salaries. The goal is to assist international job hunters in identifying job positions and companies that are more likely to sponsor H-1B visas.

**Key Findings**

1. Top Sponsoring Employers:
- Large tech companies (e.g., Amazon, Google, Meta) and consulting firms (Ernst & Young, Deloitte) dominate in terms of total certified H-1B applications.
- Tech firms generally offer higher average salaries than consulting firms.

2. Popular Job Titles by State:
- Most states show a strong preference for software-related roles, especially Software Engineer and Software Developer.
- States like CA, TX, and WA are hubs for H-1B opportunities.

3. Wage Analysis by Employer:
- By comparing median or average wages across companies, I found that some employers (e.g., Meta, Google, AWS) consistently provide higher compensation for software positions.
- A deeper salary comparison shows significant wage variation even for the same job title.

**Limitations**

Title Noise:
Job titles are not standardized, making it challenging to group or filter by seniority (e.g., distinguishing SDE I vs Senior SWE).

No Clear Entry-Level Definition:
There’s no clean label to filter only entry-level positions. Manual rule-based filtering is limited in precision.

Data Scope:
The dataset only includes certified/denied cases in 2025 (Q1–Q2), which may not fully reflect annual trends. Since the regulations just changed, I plan to wait for a period of time until the data shows a stable pattern.